## 生成式方法：WaferVAE 異常探測與稀有缺陷樣本擴增

- 目標掌握生成式模型——變分自動編碼器（VAE）的底層物理意義與數學公式（重參數化技巧）。學會利用 PyTorch 實作專門處理晶圓圖缺陷的 WaferVAE 模型，達成兩大產線核心任務：
    -   1. 稀有缺陷樣本擴增：從潛在空間（Latent Space）採樣生成全新、未曾存在但物理合理的缺陷晶圓，補足訓練集的長尾分佈缺陷。
    -   2. 無監督異常探測：利用重構潛在特徵的相似度進行異常篩選。


### 1. VAE 的核心物理意義與重參數化技巧

- 核心：
    - 與傳統 AutoEncoder（將影像壓縮成一個固定的向量）不同，變分自動編碼器 (VAE) 是將影像壓縮成一個「機率分佈」——即均值（Mean, \(\mu \)）與對數方差（Log-Variance, \(\log(\sigma^2)\)）。
    - 痛點：由於從分佈中進行隨機採樣（Sampling）是一個不可微分的動作，會導致深度學習無法進行反向傳播（Backpropagation）。
    - 重參數化技巧 (Reparameterization Trick)：引入一個獨立的正態分佈雜訊 \(\epsilon \sim N(0, I)\)，將採樣過程改寫為：\(z = \mu + \epsilon \cdot \sigma\)。如此一來，隨機性被隔絕在 \(\epsilon \)，梯度便能順利流回 \(\mu \) 與 \(\sigma \)。


### 2. WaferVAE 模型架構與 PyTorch 實作

- 實作：我們將設計一個處理 2D 晶圓圖特徵的 VAE，包含 Encoder（輸出均值與方差）、Reparameterize 採樣層、以及 Decoder（負責還原/生成晶圓圖）。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np


class WaferVAE(nn.Module):
    """對應專案：WaferVAE 異常探測與資料生成模型"""

    def __init__(self, feature_dim=1568, latent_dim=20):
        super(WaferVAE, self).__init__()

        # 編碼器：將提取的晶圓特徵映射到潛在空間的分佈參數上
        self.fc_encoder = nn.Linear(feature_dim, 128)
        self.fc_mu = nn.Linear(128, latent_dim)  # 輸出潛在空間的 均值 (mu)
        self.fc_logvar = nn.Linear(128, latent_dim)  # 輸出潛在空間的 對數方差 (logvar)

        # 解碼器：從潛在空間的點 (z) 還原回原始晶圓特徵
        self.fc_decoder_input = nn.Linear(latent_dim, 128)
        self.fc_decoder_output = nn.Linear(128, feature_dim)

    def reparameterize(self, mu, logvar):
        """核心解法：重參數化技巧"""
        if self.training:
            # 計算標準差
            std = torch.exp(0.5 * logvar)
            # 引入標準正態分佈雜訊 epsilon
            eps = torch.randn_like(std)
            # 重組 z，保留可微分的梯度鏈
            return mu + eps * std
        else:
            # 推理或資料生成時，直接使用均值，不需要雜訊
            return mu

    def forward(self, x):
        # 扁平化輸入特徵 [B, feature_dim]
        h = F.relu(self.fc_encoder(x))
        mu, logvar = self.fc_mu(h), self.fc_logvar(h)

        # 進行重參數化採樣
        z = self.reparameterize(mu, logvar)

        # 解碼還原
        h_dec = F.relu(self.fc_decoder_input(z))
        # 使用 Sigmoid 將輸出限制在 0~1 之間（符合標準化後的晶圓點位機率）
        reconstructed_x = torch.sigmoid(self.fc_decoder_output(h_dec))

        return reconstructed_x, mu, logvar

    def generate_rare_defect(self, num_samples=5):
        """任務一：從潛在空間隨機採樣，無中生有生成稀有缺陷樣本"""
        self.eval()
        with torch.no_grad():
            # 隨機生成符合標準正態分佈的潛在向量 z [num_samples, latent_dim]
            z_sampled = torch.randn(num_samples, 20)
            h_dec = F.relu(self.fc_decoder_input(z_sampled))
            generated_wafers = torch.sigmoid(self.fc_decoder_output(h_dec))
        return generated_wafers


# --- 實體化與損失函數定義 (VAE Loss = Reconstruction + KL Divergence) ---
model_vae = WaferVAE(feature_dim=1568, latent_dim=20)


def loss_function_vae(recon_x, x, mu, logvar):
    # A. 重建誤差 (Reconstruction Loss)：衡量生成的晶圓圖與真實影像是否相似
    recon_loss = F.mse_loss(recon_x, x, reduction="sum")
    # B. KL 散度 (KL Divergence)：強迫潛在空間分佈緊貼標準正態分佈，防止空間破碎
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl_loss

### 3. 稀有缺陷生成與異常偵測流水線測試

- 實作：我們來模擬如何利用受訓完成的 WaferVAE 一邊生成長尾缺陷資料，一邊檢測未知的產線異常。


In [ ]:
# 模擬產線傳入的晶圓低維特徵 (假設已由特徵提取器轉為 1568 維)
batch_size = 3
mock_real_wafer_features = torch.rand(batch_size, 1568)

# 進行前向傳播（計算損失，用於訓練階段）
recon_features, mu, logvar = model_vae(mock_real_wafer_features)
total_vae_loss = loss_function_vae(recon_features, mock_real_wafer_features, mu, logvar)
print(f"VAE 聯合損失計算成功。當前 Batch 總損失: {total_vae_loss.item():.2f}")

# 任務一展示：產線稀有缺陷樣本擴增
print("\n>>> 啟動【生成式 AI】擴增 Pipeline，無中生有合成稀有缺陷...")
new_generated_defects = model_vae.generate_rare_defect(num_samples=5)
print(
    f"成功在潛在空間採樣並生成 5 片全新的虛擬晶圓缺陷特徵。張量形狀: {new_generated_defects.shape}"
)

# 任務二展示：無監督異常探測 (Anomaly Detection)
# VAE 同樣可以計算個別樣本的重構誤差，誤差過大者即為無法辨識的全新未知異常
with torch.no_grad():
    individual_errors = torch.mean(
        (recon_features - mock_real_wafer_features) ** 2, dim=1
    )
    for idx, err in enumerate(individual_errors):
        status = "CRITICAL 異常" if err.item() > 0.15 else "✅ NORMAL 正常"
        print(
            f"Wafer #{idx + 1} 重建得分: {err.item():.4f} | 產線篩選狀態: {status}"
        )

- 總結：在半導體先進製程中，某些極特殊的製程缺陷（例如真空異常或突發性的晶圓裂痕）發生機率極低，歷史資料庫中可能只有兩三筆，這會導致傳統的 CNN 分類器因為『長尾效能不佳』而產生漏報。為了解決這個核心痛點，我引入了生成式 AI 架構，在專案中實作了 變分自動編碼器（WaferVAE）。我深入掌握了其底層的 重參數化技巧 (Reparameterization Trick)，將不可微的隨機採樣過程轉換為可傳遞梯度的線性運算，並結合 重建誤差與 KL 散度 (KL Divergence) 作為聯合損失函數，確保潛在空間（Latent Space）的分佈連續且平滑。通過受訓後的 WaferVAE，我們能直接在潛在空間進行隨機採樣，無中生有地生成物理合理、且具備多樣性的虛擬罕見缺陷樣本投餵給分類器，完美解決了數據稀缺問題。同時，它還能兼任無監督防禦，利用重建分數為未知的新型缺陷提供動態預警，這是柔性測試整合工程師在高階 AI 應用上的核心亮點。
